In [ ]:
!pip install timm -q

import os, sys, time, json, copy, random, zipfile, warnings
from pathlib import Path
from collections import defaultdict
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 150

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch : 2.10.0+cu128
Device  : cuda
GPU     : Tesla T4
VRAM    : 15.64 GB


In [ ]:
CONFIG = {
    # --- Experiment identity ---
    'experiment'    : 'saliencymix',
    'seed'          : 42,

    # --- Dataset ---
    # Options: 'cifar100' | 'tiny_imagenet' | 'cub200'
    'dataset'       : 'cifar100',
    'data_root'     : '/content/data',

    # --- Compute ---
    'batch_size'    : 128,
    'num_workers'   : 2,
    'pin_memory'    : True,

    # --- Training ---
    'epochs'        : 200,

    # --- Optimizer ---
    'lr'            : 0.1,
    'momentum'      : 0.9,
    'weight_decay'  : 1e-4,
    'nesterov'      : True,

    # --- LR Schedule ---
    'warmup_epochs' : 5,
    'lr_min'        : 1e-4,

    # --- Augmentation method ---
    # Options: 'none' | 'cutout' | 'mixup' | 'cutmix' | 'saliencymix' | 'procammix'
    'method'        : 'saliencymix',
    'alpha'         : 1.0,      # Beta distribution parameter
    'aug_prob'      : 1.0,      # Probability of applying aug per batch

    # --- Cutout specific ---
    'cutout_holes'  : 1,
    'cutout_length' : 8,        # For CIFAR-100 (32x32); use 16 for Tiny-ImageNet

    # --- ProCAMMix specific ---
    'tau_warmup'    : 20,       # Epochs before CAM guidance starts increasing
    'ema_momentum'  : 0.9,      # EMA smoothing for CAM buffer

    # --- Checkpointing ---
    'ckpt_dir'      : '/content/drive/MyDrive/ProCAMMix/checkpoints',
    'results_dir'   : '/content/drive/MyDrive/ProCAMMix/results',
    'save_every'    : 3,       # Save checkpoint every N epochs
    'resume'        : True,     # Auto-resume from latest checkpoint if exists
}

# Dataset-specific parameter overrides (applied automatically below)
_DS = {
    'cifar100'      : dict(num_classes=100, img_size=32,  cam_h=4, cam_w=4,
                           pretrained=False),
    'tiny_imagenet' : dict(num_classes=200, img_size=64,  cam_h=4, cam_w=4,
                           pretrained=False),
    'cub200'        : dict(num_classes=200, img_size=224, cam_h=7, cam_w=7,
                           pretrained=True, lr=0.01, batch_size=32,
                           epochs=100, warmup_epochs=3,
                           cutout_length=56, tau_warmup=10),
}
CONFIG.update(_DS[CONFIG['dataset']])

# Create all required directories
for d in [CONFIG['ckpt_dir'], CONFIG['results_dir'], CONFIG['data_root']]:
    os.makedirs(d, exist_ok=True)

print("Active configuration:")
for k, v in CONFIG.items():
    print(f"  {k:20s}: {v}")

Active configuration:
  experiment          : saliencymix
  seed                : 42
  dataset             : cifar100
  data_root           : /content/data
  batch_size          : 128
  num_workers         : 2
  pin_memory          : True
  epochs              : 200
  lr                  : 0.1
  momentum            : 0.9
  weight_decay        : 0.0001
  nesterov            : True
  warmup_epochs       : 5
  lr_min              : 0.0001
  method              : saliencymix
  alpha               : 1.0
  aug_prob            : 1.0
  cutout_holes        : 1
  cutout_length       : 8
  tau_warmup          : 20
  ema_momentum        : 0.9
  ckpt_dir            : /content/drive/MyDrive/ProCAMMix/checkpoints
  results_dir         : /content/drive/MyDrive/ProCAMMix/results
  save_every          : 3
  resume              : True
  num_classes         : 100
  img_size            : 32
  cam_h               : 4
  cam_w               : 4
  pretrained          : False


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(CONFIG['seed'])
print(f"Global seed: {CONFIG['seed']}")

Global seed: 42


In [ ]:
def get_cifar100(data_root, batch_size, num_workers, pin_memory):
    mean = (0.5071, 0.4867, 0.4408)
    std  = (0.2675, 0.2565, 0.2761)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    train_ds = torchvision.datasets.CIFAR100(
        data_root, train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR100(
        data_root, train=False, download=True, transform=test_tf)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=pin_memory,
                          drop_last=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)
    return train_dl, test_dl

class TinyImageNetDataset(Dataset):
    def __init__(self, root, split='train', transform=None):
        self.root      = Path(root) / 'tiny-imagenet-200'
        self.split     = split
        self.transform = transform
        self.samples   = []

        with open(self.root / 'wnids.txt') as f:
            wnids = [l.strip() for l in f]
        self.class_to_idx = {w: i for i, w in enumerate(wnids)}

        if split == 'train':
            for wnid in wnids:
                for p in (self.root / 'train' / wnid / 'images').glob('*.JPEG'):
                    self.samples.append((str(p), self.class_to_idx[wnid]))
        else:
            with open(self.root / 'val' / 'val_annotations.txt') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    label = self.class_to_idx.get(parts[1], -1)
                    if label >= 0:
                        self.samples.append((
                            str(self.root / 'val' / 'images' / parts[0]), label))

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


def get_tiny_imagenet(data_root, batch_size, num_workers, pin_memory):
    target = Path(data_root) / 'tiny-imagenet-200'
    if not target.exists():
        import urllib.request
        url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
        zip_path = Path(data_root) / 'tmp.zip'
        print("Downloading Tiny-ImageNet (~236 MB)…")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(data_root)
        zip_path.unlink()
        print("Tiny-ImageNet ready.")

    mean = (0.4802, 0.4481, 0.3975)
    std  = (0.2302, 0.2265, 0.2262)
    train_tf = transforms.Compose([
        transforms.RandomCrop(64, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    train_ds = TinyImageNetDataset(data_root, split='train', transform=train_tf)
    test_ds  = TinyImageNetDataset(data_root, split='val',   transform=test_tf)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=pin_memory,
                          drop_last=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)
    return train_dl, test_dl


# ──────────────────────────── CUB-200-2011 ───────────────────────────────────
class CUBDataset(Dataset):
    def __init__(self, root, train=True, transform=None):
        self.root      = Path(root) / 'CUB_200_2011'
        self.transform = transform
        images, labels, split_flag = {}, {}, {}

        with open(self.root / 'images.txt') as f:
            for l in f:
                i, p = l.strip().split(); images[int(i)] = p
        with open(self.root / 'image_class_labels.txt') as f:
            for l in f:
                i, c = l.strip().split(); labels[int(i)] = int(c) - 1
        with open(self.root / 'train_test_split.txt') as f:
            for l in f:
                i, s = l.strip().split(); split_flag[int(i)] = int(s)

        self.samples = [
            (str(self.root / 'images' / images[i]), labels[i])
            for i in sorted(images)
            if (train and split_flag[i] == 1) or (not train and split_flag[i] == 0)
        ]

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


def get_cub200(data_root, batch_size, num_workers, pin_memory):
    if not (Path(data_root) / 'CUB_200_2011').exists():
        raise FileNotFoundError(
            "CUB-200-2011 not found.\n"
            "Download from: https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz\n"
            f"Extract to: {data_root}  (so that {data_root}/CUB_200_2011/ exists)"
        )
    mean = (0.485, 0.456, 0.406)
    std  = (0.229, 0.224, 0.225)
    train_tf = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    train_ds = CUBDataset(data_root, train=True,  transform=train_tf)
    test_ds  = CUBDataset(data_root, train=False, transform=test_tf)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=pin_memory,
                          drop_last=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)
    return train_dl, test_dl


# ──────────────────────────────── Factory ────────────────────────────────────
def get_dataloaders(cfg):
    ds = cfg['dataset']
    kwargs = dict(batch_size=cfg['batch_size'],
                  num_workers=cfg['num_workers'],
                  pin_memory=cfg['pin_memory'])
    if ds == 'cifar100':
        return get_cifar100(cfg['data_root'], **kwargs)
    elif ds == 'tiny_imagenet':
        return get_tiny_imagenet(cfg['data_root'], **kwargs)
    elif ds == 'cub200':
        return get_cub200(cfg['data_root'], **kwargs)
    else:
        raise ValueError(f"Unknown dataset: {ds}")

print("Dataset utilities loaded.")

Dataset utilities loaded.


In [ ]:
# ──────────────────────────────── Cutout ─────────────────────────────────────
class Cutout:
    """Zero-out n_holes square regions of length x length from each image."""
    def __init__(self, n_holes: int, length: int):
        self.n_holes = n_holes
        self.length  = length

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        # img: (C, H, W)
        _, H, W = img.shape
        mask = torch.ones(H, W, dtype=img.dtype, device=img.device)
        for _ in range(self.n_holes):
            cy = random.randint(0, H - 1)
            cx = random.randint(0, W - 1)
            y1 = max(0, cy - self.length // 2)
            y2 = min(H, cy + self.length // 2)
            x1 = max(0, cx - self.length // 2)
            x2 = min(W, cx + self.length // 2)
            mask[y1:y2, x1:x2] = 0.0
        return img * mask.unsqueeze(0)


def apply_cutout(images: torch.Tensor, cfg: dict) -> torch.Tensor:
    co = Cutout(cfg['cutout_holes'], cfg['cutout_length'])
    return torch.stack([co(img) for img in images])


# ───────────────────────────────── Mixup ──────────────────────────────────────
def mixup_data(images: torch.Tensor, labels: torch.Tensor, alpha: float):
    """
    Returns: mixed_images, labels_a, labels_b, lam (scalar)
    Loss    = lam * CE(pred, labels_a) + (1-lam) * CE(pred, labels_b)
    """
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    B   = images.size(0)
    idx = torch.randperm(B, device=images.device)
    mixed = lam * images + (1.0 - lam) * images[idx]
    return mixed, labels, labels[idx], lam


# ──────────────────────────────── CutMix ─────────────────────────────────────
def _rand_bbox(H: int, W: int, lam: float):
    """Sample a random bounding box such that patch area ≈ (1-lam)*H*W."""
    cut_h = int(H * np.sqrt(1.0 - lam))
    cut_w = int(W * np.sqrt(1.0 - lam))
    cx    = random.randint(0, W)
    cy    = random.randint(0, H)
    x1 = max(0, cx - cut_w // 2);  x2 = min(W, cx + cut_w // 2)
    y1 = max(0, cy - cut_h // 2);  y2 = min(H, cy + cut_h // 2)
    return x1, y1, x2, y2


def cutmix_data(images: torch.Tensor, labels: torch.Tensor, alpha: float):
    """
    Returns: mixed_images, labels_a, labels_b, lam (scalar, area-corrected)
    """
    lam  = np.random.beta(alpha, alpha)
    B, _, H, W = images.shape
    idx  = torch.randperm(B, device=images.device)
    x1, y1, x2, y2 = _rand_bbox(H, W, lam)

    mixed      = images.clone()
    mixed[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]
    lam_actual = 1.0 - (x2 - x1) * (y2 - y1) / (H * W)   # recompute from actual box
    return mixed, labels, labels[idx], lam_actual


# ─────────────────────────────── SaliencyMix ─────────────────────────────────
def _get_saliency_map(images: torch.Tensor, labels: torch.Tensor,
                      model: nn.Module) -> torch.Tensor:
    """
    Gradient-based saliency: |d(logit_y) / d(x)|, summed across channels.
    Requires one extra forward+backward pass (no_grad disabled intentionally).
    Returns: (B, H, W) saliency maps, detached, normalized to [0,1].
    """
    imgs = images.clone().requires_grad_(True)
    logits = model(imgs)
    # Sum logits at correct class for each sample in batch
    score  = logits[range(len(labels)), labels].sum()
    model.zero_grad()
    score.backward()
    saliency = imgs.grad.data.abs().sum(dim=1)  # (B, H, W)
    # Normalize per sample
    B = saliency.shape[0]
    smin = saliency.view(B, -1).min(1)[0].view(B, 1, 1)
    smax = saliency.view(B, -1).max(1)[0].view(B, 1, 1)
    saliency = (saliency - smin) / (smax - smin + 1e-8)
    return saliency.detach()


def saliencymix_data(images: torch.Tensor, labels: torch.Tensor,
                     model: nn.Module, alpha: float):
    """
    SaliencyMix: paste the most salient region of image_j into image_i.
    Patch SIZE from Beta; patch LOCATION guided by source saliency map.
    Label ratio: area-proportional (same as CutMix — only location differs).

    Reference: Uddin et al., SaliencyMix, ICLR 2021.
    """
    lam  = np.random.beta(alpha, alpha)
    B, _, H, W = images.shape
    idx  = torch.randperm(B, device=images.device)

    # Compute saliency of source images (j)
    model.eval()
    with torch.enable_grad():
        sal_j = _get_saliency_map(images[idx], labels[idx], model)  # (B, H, W)
    model.train()

    cut_h = int(H * np.sqrt(1.0 - lam))
    cut_w = int(W * np.sqrt(1.0 - lam))

    mixed = images.clone()
    for i in range(B):
        # Find most salient center in source (j)
        s = sal_j[i]                              # (H, W)
        flat_idx = s.flatten().argmax().item()
        cy_src   = flat_idx // W
        cx_src   = flat_idx  % W

        # Destination bbox in image_i (same center, clamped)
        y1 = max(0, cy_src - cut_h // 2); y2 = min(H, y1 + cut_h)
        x1 = max(0, cx_src - cut_w // 2); x2 = min(W, x1 + cut_w)
        # Source bbox from image_j (same coordinates)
        sy1 = max(0, cy_src - cut_h // 2); sy2 = min(H, sy1 + cut_h)
        sx1 = max(0, cx_src - cut_w // 2); sx2 = min(W, sx1 + cut_w)
        # Adjust destination size to match source
        ph = min(y2 - y1, sy2 - sy1)
        pw = min(x2 - x1, sx2 - sx1)
        mixed[i, :, y1:y1+ph, x1:x1+pw] = images[idx[i], :, sy1:sy1+ph, sx1:sx1+pw]

    lam_actual = 1.0 - (cut_h * cut_w) / (H * W)
    return mixed, labels, labels[idx], lam_actual


# ──────────────────────── Unified augmentation dispatcher ────────────────────
def augment_batch(images, labels, model, cfg, epoch):
    """
    Applies the augmentation method specified in cfg['method'].
    Returns: (aug_images, labels_a, labels_b, lam)
    For 'none' and 'cutout': labels_b == labels_a, lam == 1.0.
    """
    method = cfg['method']
    alpha  = cfg['alpha']

    if method == 'none':
        return images, labels, labels, 1.0

    if method == 'cutout':
        aug = apply_cutout(images, cfg)
        return aug, labels, labels, 1.0

    if method == 'mixup':
        return mixup_data(images, labels, alpha)

    if method == 'cutmix':
        return cutmix_data(images, labels, alpha)

    if method == 'saliencymix':
        return saliencymix_data(images, labels, model, alpha)

    if method == 'procammix':
        # ProCAMMix is handled separately in Cell 6
        raise ValueError("Call procammix_data() directly for ProCAMMix.")

    raise ValueError(f"Unknown augmentation method: {method}")


print("Baseline augmentation methods loaded.")

Baseline augmentation methods loaded.


In [ ]:
# ─────────────────────── CAM Extractor (via forward hook) ────────────────────
class CAMExtractor:
    """
    Extracts Class Activation Maps from a ResNet-style model using the
    original CAM formulation (Zhou et al., 2016):

        CAM_c(x, y) = ReLU( sum_k  w_k^c * A_k(x, y) )

    where A_k is the k-th feature map of the last convolutional layer and
    w_k^c is the corresponding FC weight for class c.

    This requires GAP → FC architecture (satisfied by all ResNet variants).
    Zero additional parameters. Negligible compute overhead — features are
    captured via a forward hook during the existing inference pass.
    """
    def __init__(self, model: nn.Module):
        self.model    = model
        self._features = None
        # Register hook on the last convolutional block
        self._hook = model.layer4.register_forward_hook(self._hook_fn)

    def _hook_fn(self, module, inp, output):
        self._features = output.detach()   # (B, 2048, H_cam, W_cam)

    def get_cam(self, class_indices: torch.Tensor) -> torch.Tensor:
        """
        Args:
            class_indices: (B,) LongTensor of class indices per sample.
        Returns:
            cams: (B, H_cam, W_cam) ReLU-activated CAMs, NOT yet normalized.
        Raises:
            RuntimeError if called before a forward pass.
        """
        if self._features is None:
            raise RuntimeError("CAMExtractor: forward() has not been called yet.")
        fc_w = self.model.fc.weight          # (num_classes, 2048)
        w    = fc_w[class_indices]           # (B, 2048)
        B, C, H, W = self._features.shape
        # Weighted sum: (B, 2048, 1, 1) × (B, 2048, H, W) → sum → (B, H, W)
        cam  = (w.view(B, C, 1, 1) * self._features).sum(dim=1)
        return F.relu(cam)                   # (B, H_cam, W_cam)

    def remove_hook(self):
        self._hook.remove()


def normalize_cam(cam: torch.Tensor) -> torch.Tensor:
    """
    Normalizes each CAM to [0, 1] independently.
    Args:  cam: (B, H, W) or (H, W)
    Returns: same shape, normalized.
    """
    if cam.dim() == 2:
        cam = cam.unsqueeze(0)
        squeeze = True
    else:
        squeeze = False
    B = cam.shape[0]
    vmin = cam.view(B, -1).min(1)[0].view(B, 1, 1)
    vmax = cam.view(B, -1).max(1)[0].view(B, 1, 1)
    cam  = (cam - vmin) / (vmax - vmin + 1e-8)
    return cam.squeeze(0) if squeeze else cam


# ────────────────────────── EMA CAM Buffer ───────────────────────────────────
class EMACAMBuffer:
    """
    Per-class Exponential Moving Average buffer for CAM maps.
    Accumulates reliable, smooth CAM representations for each class
    across training batches. Reduces intra-batch CAM noise.

    Buffer entries are initialized lazily on first observation of each class.
    For classes not yet observed, falls back to a uniform (all-ones) map.
    """
    def __init__(self, num_classes: int, cam_h: int, cam_w: int,
                 momentum: float = 0.9):
        self.momentum    = momentum
        self.cam_h       = cam_h
        self.cam_w       = cam_w
        self._buffer     = torch.zeros(num_classes, cam_h, cam_w)   # CPU
        self._init_mask  = torch.zeros(num_classes, dtype=torch.bool)

    @torch.no_grad()
    def update(self, cams: torch.Tensor, labels: torch.Tensor):
        """
        Args:
            cams  : (B, H_cam, W_cam) normalized CAMs (CPU tensor)
            labels: (B,) LongTensor of class indices (CPU)
        """
        for i in range(len(labels)):
            c   = labels[i].item()
            cam = cams[i]
            if not self._init_mask[c]:
                self._buffer[c]    = cam.clone()
                self._init_mask[c] = True
            else:
                self._buffer[c] = (self.momentum * self._buffer[c]
                                   + (1.0 - self.momentum) * cam)

    def get(self, class_idx: int) -> torch.Tensor:
        """Returns the EMA CAM for class_idx, or a uniform map if unseen."""
        if not self._init_mask[class_idx]:
            return torch.ones(self.cam_h, self.cam_w)   # uniform fallback
        return self._buffer[class_idx]


# ──────────────────────── Progressive Schedule ────────────────────────────────
def get_tau(epoch: int, total_epochs: int, warmup_epochs: int) -> float:
    """
    Computes the progressive annealing coefficient τ ∈ [0, 1].

    τ = 0 during the first `warmup_epochs` epochs (pure random CutMix
        behaviour — CAMs unreliable at initialization).
    τ → 1 linearly from epoch `warmup_epochs` to `total_epochs`.

    Design rationale: CAMs trained with random weight initialization are
    noisy and background-biased. The progressive schedule exploits CAM
    reliability which monotonically improves with training depth.
    """
    if epoch < warmup_epochs:
        return 0.0
    return float(epoch - warmup_epochs) / float(total_epochs - warmup_epochs)


# ──────────────────────── CAM-guided BBox Sampling ────────────────────────────
def _cam_guided_bbox(cam: torch.Tensor, H: int, W: int,
                     lam: float, tau: float):
    """
    Samples a bounding box with center drawn from:
        p(cx, cy) ∝ τ * cam_upsampled(cx, cy) + (1-τ) * Uniform

    Args:
        cam  : (H_cam, W_cam) normalized CAM [0,1] for the SOURCE image (j).
        H, W : Target image spatial dimensions.
        lam  : Beta-distributed mixing coefficient (controls patch size).
        tau  : Progressive coefficient (0=random, 1=CAM-guided).
    Returns:
        (x1, y1, x2, y2): Bounding box in image coordinates.
    """
    cut_h = int(H * np.sqrt(max(0.0, 1.0 - lam)))
    cut_w = int(W * np.sqrt(max(0.0, 1.0 - lam)))
    cut_h = max(1, cut_h);  cut_w = max(1, cut_w)

    if tau < 1e-6 or cam.sum() < 1e-6:
        # Pure random (identical to CutMix)
        cy = random.randint(0, H - 1)
        cx = random.randint(0, W - 1)
    else:
        # Upsample CAM to image resolution
        cam_up = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0).float(),
            size=(H, W), mode='bilinear', align_corners=False
        ).squeeze()                              # (H, W)
        # Blend with uniform distribution based on tau
        blended = tau * cam_up + (1.0 - tau) * torch.ones_like(cam_up)
        blended = blended / blended.sum()
        # Sample center from blended probability distribution
        flat    = blended.flatten().cpu().numpy().astype(np.float64)
        flat   /= flat.sum()                    # re-normalize for numerical stability
        idx     = np.random.choice(len(flat), p=flat)
        cy, cx  = divmod(idx, W)

    x1 = max(0, cx - cut_w // 2);  x2 = min(W, x1 + cut_w)
    y1 = max(0, cy - cut_h // 2);  y2 = min(H, y1 + cut_h)
    # Adjust if clamping reduced size
    x2 = min(W, x1 + cut_w);  y2 = min(H, y1 + cut_h)
    return x1, y1, x2, y2


# ──────────────────────── ProCAMMix Core Function ────────────────────────────
def procammix_data(images:      torch.Tensor,
                   labels:      torch.Tensor,
                   model:       nn.Module,
                   cam_extractor: CAMExtractor,
                   ema_buffer:  EMACAMBuffer,
                   tau:         float,
                   alpha:       float = 1.0):
    """
    ProCAMMix augmentation.

    Pipeline (per batch):
    1. [No-grad forward] Extract CAMs for current batch via the CAM extractor.
    2. Normalize CAMs; update EMA buffer with current-batch CAMs.
    3. For each sample i in the batch:
       a. Sample pair index j = randperm[i].
       b. Sample λ ~ Beta(α, α).
       c. Retrieve EMA CAM for class j from the buffer.
       d. Sample bounding box center from CAM_j-guided distribution (τ-blended).
       e. Paste pixels from images[j] into images[i] at the sampled bbox.
       f. Compute semantic label correction:
            λ_area     = remaining-i-area / total-area          (CutMix baseline)
            λ_cam_mass = 1 - CAM_j[patch].sum() / CAM_j.sum()  (semantic)
            λ_semantic = (1-τ) * λ_area + τ * λ_cam_mass
    4. Return mixed batch and per-sample λ tensors.

    Loss computation (in training loop):
        loss = (λ * CE(pred, y_i) + (1-λ) * CE(pred, y_j)).mean()

    Args:
        images       : (B, C, H, W) CUDA tensor.
        labels       : (B,) LongTensor, CUDA.
        model        : Training model (must be in train mode).
        cam_extractor: Registered on model.layer4.
        ema_buffer   : Per-class EMA CAM buffer.
        tau          : Progressive coefficient ∈ [0,1].
        alpha        : Beta distribution parameter.

    Returns:
        mixed_images : (B, C, H, W)
        labels_a     : (B,) — original labels (y_i)
        labels_b     : (B,) — paired labels   (y_j)
        lam          : (B,) — per-sample semantic λ (proportion from image i)
    """
    B, C, H, W = images.shape

    # ── Step 1: Extract CAMs (no gradient) ───────────────────────────────────
    model.eval()
    with torch.no_grad():
        _      = model(images)                                  # populate hook
        cams   = cam_extractor.get_cam(labels)                  # (B, H_cam, W_cam)
        cams_n = normalize_cam(cams)                            # (B, H_cam, W_cam)
    model.train()

    # ── Step 2: Update EMA buffer ─────────────────────────────────────────────
    ema_buffer.update(cams_n.cpu(), labels.cpu())

    # ── Step 3: Construct mixed batch ────────────────────────────────────────
    idx          = torch.randperm(B, device=images.device)
    mixed_images = images.clone()
    labels_a     = labels
    labels_b     = labels[idx]
    lam_list     = []

    for i in range(B):
        j   = idx[i].item()
        lam = float(np.random.beta(alpha, alpha))

        # Retrieve EMA CAM for SOURCE class (j) — guides patch placement
        cam_j = ema_buffer.get(labels_b[i].item())             # (H_cam, W_cam), CPU

        # Sample bbox using CAM_j as probability distribution
        x1, y1, x2, y2 = _cam_guided_bbox(cam_j, H, W, lam, tau)

        # Paste pixels from j into i
        mixed_images[i, :, y1:y2, x1:x2] = images[j, :, y1:y2, x1:x2]

        # ── Step 3f: Semantic label correction ───────────────────────────────
        patch_area_ratio = (x2 - x1) * (y2 - y1) / (H * W + 1e-8)

        # CAM mass of j that falls within the pasted patch
        cam_j_up = F.interpolate(
            cam_j.unsqueeze(0).unsqueeze(0).float(),
            size=(H, W), mode='bilinear', align_corners=False
        ).squeeze()                                             # (H, W)
        cam_j_total       = cam_j_up.sum().item() + 1e-8
        cam_j_in_patch    = cam_j_up[y1:y2, x1:x2].sum().item()
        cam_mass_ratio_j  = cam_j_in_patch / cam_j_total       # semantic weight of j

        # λ = proportion of sample i contribution (area-based at τ=0, semantic at τ=1)
        lam_area     = 1.0 - patch_area_ratio
        lam_semantic = 1.0 - cam_mass_ratio_j
        lam_final    = (1.0 - tau) * lam_area + tau * lam_semantic
        lam_final    = float(np.clip(lam_final, 0.0, 1.0))
        lam_list.append(lam_final)

    lam_tensor = torch.tensor(lam_list, dtype=torch.float32, device=images.device)
    return mixed_images, labels_a, labels_b, lam_tensor


print("ProCAMMix implementation loaded.")

ProCAMMix implementation loaded.


In [ ]:
def build_model(cfg: dict) -> nn.Module:
    """
    Builds ResNet-50 with modifications appropriate for each dataset.

    CIFAR-100 / Tiny-ImageNet (small images):
        - Replace 7×7 conv (stride 2) → 3×3 conv (stride 1)
        - Remove MaxPool (would collapse 32×32 feature maps to 8×8 too early)
        This is standard practice in the augmentation literature for CIFAR-scale
        inputs (Yun et al., 2019; Kim et al., 2020).

    CUB-200-2011 (224×224 images):
        - Standard ResNet-50 architecture unchanged.
        - ImageNet pretrained weights (torchvision default).
        - Replace FC layer for num_classes=200.
    """
    num_classes = cfg['num_classes']
    pretrained  = cfg.get('pretrained', False)

    model = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)

    if cfg['dataset'] in ('cifar100', 'tiny_imagenet'):
        # Modify for small spatial inputs
        model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1,
                                  padding=1, bias=False)
        model.maxpool = nn.Identity()

    # Replace final FC for target number of classes
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    # Weight init for the new FC (Kaiming uniform)
    nn.init.kaiming_uniform_(model.fc.weight, mode='fan_out', nonlinearity='relu')
    if model.fc.bias is not None:
        nn.init.zeros_(model.fc.bias)

    return model.to(DEVICE)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = build_model(CONFIG)
print(f"Model      : ResNet-50 ({'pretrained' if CONFIG.get('pretrained') else 'scratch'})")
print(f"Dataset    : {CONFIG['dataset']} ({CONFIG['num_classes']} classes)")
print(f"Parameters : {count_parameters(model):,}")

Model      : ResNet-50 (scratch)
Dataset    : cifar100 (100 classes)
Parameters : 23,705,252


In [ ]:
# ────────────────────────── Metric utilities ─────────────────────────────────
class AverageMeter:
    """Tracks running average of a metric."""
    def __init__(self): self.reset()
    def reset(self):    self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val   = val
        self.sum  += val * n
        self.count += n
        self.avg   = self.sum / self.count


@torch.no_grad()
def accuracy(output: torch.Tensor, target: torch.Tensor,
             topk=(1, 5)) -> list:
    """Returns top-k accuracy values as Python floats (percentage)."""
    maxk = max(topk)
    B    = target.size(0)
    _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
    pred     = pred.t()
    correct  = pred.eq(target.view(1, -1).expand_as(pred))
    res = []
    for k in topk:
        c = correct[:k].reshape(-1).float().sum().item()
        res.append(100.0 * c / B)
    return res


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    """
    Computes mixed cross-entropy loss for augmented samples.
    Handles both scalar lam (Mixup/CutMix) and per-sample lam (ProCAMMix).

    Args:
        criterion : nn.CrossEntropyLoss(reduction='none') or 'mean'.
        pred      : (B, num_classes) logits.
        y_a, y_b  : (B,) label tensors.
        lam       : scalar float or (B,) tensor.
    Returns:
        Scalar loss.
    """
    loss_a = F.cross_entropy(pred, y_a, reduction='none')   # (B,)
    loss_b = F.cross_entropy(pred, y_b, reduction='none')   # (B,)
    if isinstance(lam, torch.Tensor):
        loss = (lam * loss_a + (1.0 - lam) * loss_b).mean()
    else:
        loss = lam * loss_a.mean() + (1.0 - lam) * loss_b.mean()
    return loss


# ──────────────────────── Optimizer & Scheduler ───────────────────────────────
def build_optimizer(model: nn.Module, cfg: dict) -> optim.Optimizer:
    return optim.SGD(
        model.parameters(),
        lr=cfg['lr'],
        momentum=cfg['momentum'],
        weight_decay=cfg['weight_decay'],
        nesterov=cfg['nesterov'],
    )


def build_scheduler(optimizer: optim.Optimizer,
                           cfg: dict) -> optim.lr_scheduler.LambdaLR:
    """
    Single LambdaLR encoding warmup + cosine decay in one closed-form function.

    Replaces SequentialLR entirely. Advantage: load_state_dict() restores
    a single integer (last_epoch) with no sub-scheduler state — eliminating
    the SequentialLR resume bug that caused LR to restart mid-training.

    LR schedule:
        epoch < warmup_epochs : linear warmup from 0 → base_lr
        epoch >= warmup_epochs: cosine decay from base_lr → lr_min
    """
    total   = cfg['epochs']
    warmup  = cfg['warmup_epochs']
    lr_min  = cfg['lr_min']
    lr_max  = cfg['lr']

    def lr_lambda(epoch: int) -> float:
        if epoch < warmup:
            # Linear warmup: fraction of base lr
            return (epoch + 1) / warmup
        # Cosine decay phase
        progress      = (epoch - warmup) / max(1, total - warmup)
        cosine_factor = 0.5 * (1.0 + math.cos(math.pi * progress))
        # Return as fraction of base_lr (LambdaLR multiplies by base_lr)
        return lr_min / lr_max + (1.0 - lr_min / lr_max) * cosine_factor

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ────────────────────────── Checkpointing ────────────────────────────────────
def save_checkpoint(state: dict, path: str):
    torch.save(state, path)

def load_checkpoint(path: str, model: nn.Module, optimizer=None,
                    scheduler=None):
    ckpt      = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    if optimizer  and 'optimizer'  in ckpt: optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler  and 'scheduler'  in ckpt: scheduler.load_state_dict(ckpt['scheduler'])
    return ckpt.get('epoch', 0), ckpt.get('best_acc', 0.0), ckpt.get('history', [])

def get_latest_checkpoint(ckpt_dir: str, prefix: str) -> str | None:
    files = sorted(Path(ckpt_dir).glob(f"{prefix}_epoch*.pt"))
    return str(files[-1]) if files else None


print("Training utilities loaded.")

Training utilities loaded.


In [ ]:
def train_one_epoch(model, train_dl, optimizer, cfg, epoch,
                    cam_extractor=None, ema_buffer=None) -> dict:
    """
    Runs one training epoch.

    For methods other than ProCAMMix, cam_extractor and ema_buffer
    are ignored. For SaliencyMix, temporary model.eval()/train() calls
    occur inside the augmentation function.

    Returns: dict with keys 'loss', 'top1', 'top5', 'lr'
    """
    model.train()
    loss_m  = AverageMeter()
    top1_m  = AverageMeter()
    top5_m  = AverageMeter()
    method  = cfg['method']
    tau     = get_tau(epoch, cfg['epochs'], cfg['tau_warmup'])

    for batch_idx, (images, labels) in enumerate(train_dl):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        B      = images.size(0)

        # ── Augmentation ──────────────────────────────────────────────────────
        if method == 'procammix':
            mixed, y_a, y_b, lam = procammix_data(
                images, labels, model, cam_extractor, ema_buffer, tau, cfg['alpha'])
        elif method in ('none', 'cutout', 'mixup', 'cutmix', 'saliencymix'):
            mixed, y_a, y_b, lam = augment_batch(images, labels, model, cfg, epoch)
        else:
            raise ValueError(f"Unknown method: {method}")

        # ── Forward pass ──────────────────────────────────────────────────────
        optimizer.zero_grad()
        logits = model(mixed)
        loss   = mixed_criterion(None, logits, y_a, y_b, lam)

        # ── Backward pass ─────────────────────────────────────────────────────
        loss.backward()
        # Gradient clipping (prevents rare instability in early training)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        # ── Metrics (on ORIGINAL labels for interpretability) ─────────────────
        with torch.no_grad():
            acc1, acc5 = accuracy(logits, labels, topk=(1, 5))
        loss_m.update(loss.item(), B)
        top1_m.update(acc1, B)
        top5_m.update(acc5, B)

        if (batch_idx + 1) % 100 == 0:
            print(f"  [{epoch:3d}][{batch_idx+1:4d}/{len(train_dl)}] "
                  f"loss={loss_m.avg:.4f}  top1={top1_m.avg:.2f}%  τ={tau:.3f}")

    return dict(loss=loss_m.avg, top1=top1_m.avg, top5=top5_m.avg,
                lr=optimizer.param_groups[0]['lr'])


@torch.no_grad()
def evaluate(model, test_dl) -> dict:
    """Evaluates model on test set. Returns dict with top1, top5."""
    model.eval()
    top1_m = AverageMeter()
    top5_m = AverageMeter()
    for images, labels in test_dl:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        acc1, acc5 = accuracy(logits, labels, topk=(1, 5))
        top1_m.update(acc1, images.size(0))
        top5_m.update(acc5, images.size(0))
    model.train()
    return dict(top1=top1_m.avg, top5=top5_m.avg)


print("Training and evaluation loops loaded.")

Training and evaluation loops loaded.


In [ ]:
# =============================================================================
# CELL 10 — Main Training Loop
# This cell executes the complete training run.
# Auto-resumes from the latest checkpoint if cfg['resume'] is True.
# Expected runtime on T4 GPU:
#   CIFAR-100  : ~3–4 hours  (200 epochs, ResNet-50, BS=128)
#   Tiny-IN    : ~6–8 hours  (200 epochs, ResNet-50, BS=128)
#   CUB-200    : ~2–3 hours  (100 epochs, ResNet-50, BS=32)
# IMPORTANT: Connect Colab to T4 before running. Save checkpoints to Drive.
# =============================================================================

def run_experiment(cfg: dict):
    print(f"\n{'='*65}")
    print(f"  Experiment : {cfg['experiment']}")
    print(f"  Dataset    : {cfg['dataset']}")
    print(f"  Method     : {cfg['method']}")
    print(f"  Epochs     : {cfg['epochs']}")
    print(f"{'='*65}\n")
    set_seed(cfg['seed'])

    # ── Data ──────────────────────────────────────────────────────────────────
    train_dl, test_dl = get_dataloaders(cfg)
    print(f"Train batches: {len(train_dl)} | Test batches: {len(test_dl)}")

    # ── Model ─────────────────────────────────────────────────────────────────
    model     = build_model(cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)

    # ── ProCAMMix infrastructure ──────────────────────────────────────────────
    cam_extractor = None
    ema_buffer    = None
    if cfg['method'] == 'procammix':
        cam_extractor = CAMExtractor(model)
        ema_buffer    = EMACAMBuffer(
            num_classes = cfg['num_classes'],
            cam_h       = cfg['cam_h'],
            cam_w       = cfg['cam_w'],
            momentum    = cfg['ema_momentum'],
        )

    # ── Resume from checkpoint ────────────────────────────────────────────────
    start_epoch = 0
    best_acc    = 0.0
    history     = []
    prefix      = f"{cfg['experiment']}_{cfg['dataset']}_{cfg['method']}"

    if cfg['resume']:
      ckpt_path = get_latest_checkpoint(cfg['ckpt_dir'], prefix)

      # CRITICAL: Load the BEST checkpoint, not the latest.
      # The latest checkpoint (epoch 147) has corrupted weights.
      # Force-load the best checkpoint by name.
      best_ckpt = os.path.join(
          cfg['ckpt_dir'],
          f"{prefix}_best.pt"
      )
      ckpt_to_load = best_ckpt if os.path.exists(best_ckpt) else ckpt_path

      if ckpt_to_load and os.path.exists(ckpt_to_load):
          ckpt         = torch.load(ckpt_to_load, map_location=DEVICE)
          model.load_state_dict(ckpt['model'])
          start_epoch  = ckpt.get('epoch', 0)
          best_acc     = ckpt.get('best_acc', 0.0)
          history      = ckpt.get('history', [])

          # Restore optimizer state
          optimizer.load_state_dict(ckpt['optimizer'])

          # DO NOT restore scheduler state dict.
          # DO NOT re-step scheduler N times.
          # LambdaLR computes correct LR from current epoch number automatically.
          # Manually set last_epoch so scheduler.step() begins from correct epoch.
          scheduler.last_epoch = start_epoch - 1

          # Verify the LR is correct before training resumes
          scheduler.step()
          current_lr = optimizer.param_groups[0]['lr']
          print(f"Resumed from : {ckpt_to_load}")
          print(f"Resume epoch : {start_epoch}")
          print(f"Best acc     : {best_acc:.2f}%")
          print(f"Restored LR  : {current_lr:.6f}  (must be ~0.002–0.003)")
      else:
          print("No checkpoint found — starting from scratch.")
      # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(start_epoch, cfg['epochs']):
        t0 = time.time()

        train_metrics = train_one_epoch(
            model, train_dl, optimizer, cfg, epoch,
            cam_extractor, ema_buffer
        )
        scheduler.step()
        test_metrics  = evaluate(model, test_dl)

        elapsed = time.time() - t0
        is_best = test_metrics['top1'] > best_acc
        if is_best:
            best_acc = test_metrics['top1']

        record = dict(
            epoch      = epoch,
            train_loss = round(train_metrics['loss'],  4),
            train_top1 = round(train_metrics['top1'],  2),
            test_top1  = round(test_metrics['top1'],   2),
            test_top5  = round(test_metrics['top5'],   2),
            lr         = round(train_metrics['lr'],     6),
            best_acc   = round(best_acc, 2),
            tau        = round(get_tau(epoch, cfg['epochs'], cfg['tau_warmup']), 4),
            time_s     = round(elapsed, 1),
        )
        history.append(record)

        print(f"Epoch {epoch:3d}/{cfg['epochs']-1} | "
              f"loss={record['train_loss']:.4f} | "
              f"train={record['train_top1']:.2f}% | "
              f"test={record['test_top1']:.2f}% | "
              f"best={record['best_acc']:.2f}% | "
              f"τ={record['tau']:.3f} | "
              f"lr={record['lr']:.5f} | "
              f"t={elapsed:.0f}s"
              + (" ← BEST" if is_best else ""))

        # Save checkpoint every N epochs and on best
        if (epoch + 1) % cfg['save_every'] == 0 or is_best:
            ckpt_state = dict(
                epoch     = epoch + 1,
                model     = model.state_dict(),
                optimizer = optimizer.state_dict(),
                scheduler = scheduler.state_dict(),
                best_acc  = best_acc,
                history   = history,
                config    = cfg,
            )
            tag  = 'best' if is_best else f'epoch{epoch+1:03d}'
            path = os.path.join(cfg['ckpt_dir'], f"{prefix}_{tag}.pt")
            save_checkpoint(ckpt_state, path)

    # ── Save history JSON ─────────────────────────────────────────────────────
    hist_path = os.path.join(cfg['results_dir'], f"{prefix}_history.json")
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"\nTraining complete. Best test accuracy: {best_acc:.2f}%")
    print(f"History saved to: {hist_path}")

    if cam_extractor:
        cam_extractor.remove_hook()

    return history, best_acc


# ── Execute ────────────────────────────────────────────────────────────────────
# IMPORTANT: Run one method at a time.
# After each run, change CONFIG['method'] in Cell 2, re-run Cells 2–9, then Cell 10.
# Results from all runs are aggregated in Cell 11.
history, best_acc = run_experiment(CONFIG)


  Experiment : saliencymix
  Dataset    : cifar100
  Method     : saliencymix
  Epochs     : 200

Train batches: 390 | Test batches: 79
Resumed from : /content/drive/MyDrive/ProCAMMix/checkpoints/saliencymix_cifar100_saliencymix_epoch155.pt
Resume epoch : 155
Best acc     : 79.46%
Restored LR  : 0.012662  (must be ~0.002–0.003)
  [155][ 100/390] loss=0.9566  top1=50.24%  τ=0.750
  [155][ 200/390] loss=0.9826  top1=46.62%  τ=0.750
  [155][ 300/390] loss=0.9749  top1=46.63%  τ=0.750
Epoch 155/199 | loss=0.9730 | train=46.94% | test=79.41% | best=79.46% | τ=0.750 | lr=0.01266 | t=334s
  [156][ 100/390] loss=0.9652  top1=44.58%  τ=0.756
  [156][ 200/390] loss=0.9644  top1=46.13%  τ=0.756
  [156][ 300/390] loss=0.9547  top1=48.60%  τ=0.756
Epoch 156/199 | loss=0.9445 | train=50.31% | test=79.84% | best=79.84% | τ=0.756 | lr=0.01213 | t=334s ← BEST
  [157][ 100/390] loss=0.9433  top1=52.30%  τ=0.761
  [157][ 200/390] loss=0.9647  top1=52.16%  τ=0.761
  [157][ 300/390] loss=0.9615  top1=53.1

In [ ]:
# =============================================================================
# CELL 11 — Results Aggregation and Paper-Ready Table Generator
# Run this cell AFTER completing all method runs.
# It reads all history JSON files from results_dir and builds the final table.
# =============================================================================

import json, os, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = CONFIG['results_dir']
DATASET     = CONFIG['dataset']

# ── Load all results ──────────────────────────────────────────────────────────
results = {}
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, f"*_{DATASET}_*.json"))):
    method_name = path.split(f'_{DATASET}_')[1].replace('_history.json','')
    with open(path) as f:
        h = json.load(f)
    # Extract best test accuracy from history
    best_top1 = max(r['test_top1'] for r in h)
    best_top5 = max(r['test_top5'] for r in h) if 'test_top5' in h[0] else 0.0
    final_top1 = h[-1]['test_top1']
    results[method_name] = dict(best_top1=best_top1, best_top5=best_top5,
                                final_top1=final_top1, history=h)

# ── Method display names (for paper table) ────────────────────────────────────
METHOD_LABELS = {
    'none'        : 'Baseline (No Augmentation)',
    'cutout'      : 'Cutout \\cite{devries2017improved}',
    'mixup'       : 'Mixup \\cite{zhang2018mixup}',
    'cutmix'      : 'CutMix \\cite{yun2019cutmix}',
    'saliencymix' : 'SaliencyMix \\cite{uddin2021saliencymix}',
    'procammix'   : '\\textbf{ProCAMMix (Ours)}',
}

# ── Console Table ─────────────────────────────────────────────────────────────
print(f"\n{'─'*65}")
print(f"  Results on {DATASET.upper():<15s}  ResNet-50  200 Epochs")
print(f"{'─'*65}")
print(f"  {'Method':<38s}  {'Top-1':>6s}  {'Top-5':>6s}")
print(f"{'─'*65}")
PREFERRED_ORDER = ['none','cutout','mixup','cutmix','saliencymix','procammix']
for m in PREFERRED_ORDER:
    if m in results:
        r = results[m]
        mark = ' ← Proposed' if m == 'procammix' else ''
        print(f"  {m:<38s}  {r['best_top1']:>6.2f}  {r['best_top5']:>6.2f}{mark}")
print(f"{'─'*65}\n")

# ── LaTeX Table (copy directly into paper) ────────────────────────────────────
print("% ── LaTeX Table ──────────────────────────────────────────────────────")
print("\\begin{table}[t]")
print("\\centering")
print("\\caption{Top-1 and Top-5 classification accuracy (\\%) on %s with ResNet-50." % DATASET.upper())
print("All methods trained for 200 epochs with identical hyperparameters.")
print("Best results highlighted in \\textbf{bold}.}")
print("\\label{tab:main_results}")
print("\\begin{tabular}{lcc}")
print("\\toprule")
print("Method & Top-1 (\\%) & Top-5 (\\%) \\\\")
print("\\midrule")
best_top1 = max(r['best_top1'] for r in results.values())
for m in PREFERRED_ORDER:
    if m in results:
        r   = results[m]
        t1  = f"\\textbf{{{r['best_top1']:.2f}}}" if r['best_top1'] == best_top1 else f"{r['best_top1']:.2f}"
        t5  = f"{r['best_top5']:.2f}"
        lab = METHOD_LABELS.get(m, m)
        print(f"{lab} & {t1} & {t5} \\\\")
print("\\bottomrule")
print("\\end{tabular}")
print("\\end{table}")

# ── Training Curves Plot (for paper Figure) ───────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = {'none':'gray','cutout':'brown','mixup':'blue','cutmix':'green',
          'saliencymix':'orange','procammix':'red'}
lws    = {'procammix': 2.5}

for m in PREFERRED_ORDER:
    if m not in results: continue
    h  = results[m]['history']
    ep = [r['epoch']      for r in h]
    tr = [r['train_top1'] for r in h]
    te = [r['test_top1']  for r in h]
    c  = colors.get(m, 'black')
    lw = lws.get(m, 1.5)
    ls = '-' if m == 'procammix' else '--'
    lab = m if m != 'procammix' else 'ProCAMMix (Ours)'
    ax1.plot(ep, tr, color=c, lw=lw, ls=ls, label=lab, alpha=0.85)
    ax2.plot(ep, te, color=c, lw=lw, ls=ls, label=lab, alpha=0.85)

for ax, title in zip([ax1, ax2], ['Training Accuracy', 'Test Accuracy']):
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Top-1 Accuracy (%)', fontsize=12)
    ax.set_title(f'{DATASET.upper()} — {title}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, f'training_curves_{DATASET}.pdf')
plt.savefig(fig_path, bbox_inches='tight', dpi=300)
plt.show()
print(f"\nFigure saved to: {fig_path}")

In [ ]:
# =============================================================================
# CELL 12 — Ablation Study
# Runs the three ablation conditions that are required by Q1 reviewers.
# Run AFTER the main ProCAMMix run completes.
#
# Ablation design:
#   A1. ProCAMMix without progressive schedule (τ fixed to 1 from epoch 0)
#       → Tests: is the warmup necessary?
#   A2. ProCAMMix without semantic label correction (area-based only, τ_label=0)
#       → Tests: does label correction contribute independently?
#   A3. ProCAMMix without EMA buffer (use raw batch CAMs, no smoothing)
#       → Tests: does EMA smoothing improve stability?
# =============================================================================

# ── Ablation variant definitions ──────────────────────────────────────────────
# Each variant inherits the full ProCAMMix CONFIG and overrides specific settings.

def procammix_ablation_data(images, labels, model, cam_extractor,
                             ema_buffer, tau, alpha,
                             disable_progressive=False,
                             disable_label_correction=False,
                             disable_ema=False):
    """
    Ablated version of procammix_data.
    Flags:
        disable_progressive     : τ=1 from epoch 0 (no warmup)
        disable_label_correction: use area-based λ only (τ_label=0)
        disable_ema             : use raw single-batch CAMs, no EMA
    """
    if disable_progressive:
        tau = 1.0

    B, C, H, W = images.shape
    model.eval()
    with torch.no_grad():
        _      = model(images)
        cams   = cam_extractor.get_cam(labels)
        cams_n = normalize_cam(cams)
    model.train()

    if not disable_ema:
        ema_buffer.update(cams_n.cpu(), labels.cpu())

    idx          = torch.randperm(B, device=images.device)
    mixed_images = images.clone()
    labels_a     = labels
    labels_b     = labels[idx]
    lam_list     = []

    for i in range(B):
        j   = idx[i].item()
        lam = float(np.random.beta(alpha, alpha))

        if disable_ema:
            # Use raw current-batch CAM instead of EMA buffer
            cam_j = cams_n[j].cpu()
        else:
            cam_j = ema_buffer.get(labels_b[i].item())

        x1, y1, x2, y2 = _cam_guided_bbox(cam_j, H, W, lam, tau)
        mixed_images[i, :, y1:y2, x1:x2] = images[j, :, y1:y2, x1:x2]

        patch_area_ratio = (x2 - x1) * (y2 - y1) / (H * W + 1e-8)
        lam_area = 1.0 - patch_area_ratio

        if disable_label_correction:
            lam_final = lam_area
        else:
            cam_j_up  = F.interpolate(
                cam_j.unsqueeze(0).unsqueeze(0).float(),
                size=(H, W), mode='bilinear', align_corners=False
            ).squeeze()
            cam_mass = cam_j_up[y1:y2, x1:x2].sum() / (cam_j_up.sum() + 1e-8)
            lam_semantic = 1.0 - cam_mass.item()
            lam_final    = (1.0 - tau) * lam_area + tau * lam_semantic

        lam_list.append(float(np.clip(lam_final, 0.0, 1.0)))

    lam_tensor = torch.tensor(lam_list, dtype=torch.float32, device=images.device)
    return mixed_images, labels_a, labels_b, lam_tensor


# ── Ablation execution helper ─────────────────────────────────────────────────
ABLATION_CONFIGS = {
    'ProCAMMix_A1_NoProgressive' : dict(disable_progressive=True,
                                        disable_label_correction=False,
                                        disable_ema=False),
    'ProCAMMix_A2_NoLabelCorr'   : dict(disable_progressive=False,
                                        disable_label_correction=True,
                                        disable_ema=False),
    'ProCAMMix_A3_NoEMA'         : dict(disable_progressive=False,
                                        disable_label_correction=False,
                                        disable_ema=True),
}
# NOTE: Run each ablation by modifying the run_experiment function to call
# procammix_ablation_data instead of procammix_data, with the appropriate flags.
# A convenience wrapper is provided here for direct use:

def run_ablation(cfg, ablation_name, ablation_flags):
    """Run a single ablation variant. Mirrors run_experiment but calls ablated fn."""
    print(f"\n[ABLATION] {ablation_name}")
    set_seed(cfg['seed'])
    train_dl, test_dl = get_dataloaders(cfg)
    model     = build_model(cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)
    cam_extractor = CAMExtractor(model)
    ema_buffer    = EMACAMBuffer(cfg['num_classes'], cfg['cam_h'],
                                 cfg['cam_w'], cfg['ema_momentum'])
    history = []; best_acc = 0.0

    for epoch in range(cfg['epochs']):
        model.train()
        loss_m = AverageMeter(); top1_m = AverageMeter()
        tau    = get_tau(epoch, cfg['epochs'], cfg['tau_warmup'])

        for images, labels in train_dl:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            mixed, y_a, y_b, lam = procammix_ablation_data(
                images, labels, model, cam_extractor, ema_buffer, tau,
                cfg['alpha'], **ablation_flags)
            optimizer.zero_grad()
            logits = model(mixed)
            loss   = mixed_criterion(None, logits, y_a, y_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            with torch.no_grad():
                a1, _ = accuracy(logits, labels)
            loss_m.update(loss.item(), images.size(0))
            top1_m.update(a1, images.size(0))

        scheduler.step()
        test_m = evaluate(model, test_dl)
        if test_m['top1'] > best_acc: best_acc = test_m['top1']

        if epoch % 20 == 0 or epoch == cfg['epochs'] - 1:
            print(f"  Epoch {epoch:3d} | test={test_m['top1']:.2f}% | best={best_acc:.2f}%")

        history.append(dict(epoch=epoch, test_top1=test_m['top1'],
                            train_top1=top1_m.avg, train_loss=loss_m.avg))

    path = os.path.join(cfg['results_dir'],
                        f"{ablation_name}_{cfg['dataset']}_history.json")
    with open(path, 'w') as f: json.dump(history, f, indent=2)
    cam_extractor.remove_hook()
    print(f"  Best accuracy: {best_acc:.2f}% | saved to {path}")
    return history, best_acc

# ── Run ablations ─────────────────────────────────────────────────────────────
# Uncomment and run after the full ProCAMMix experiment is complete:
#
for name, flags in ABLATION_CONFIGS.items():
     run_ablation(CONFIG, name, flags)
print("Ablation utilities loaded. Uncomment the loop above to run ablations.")

NameError: name 'CONFIG' is not defined

In [ ]:
# =============================================================================
# CELL 13 — τ (Tau) Sensitivity Analysis
# Validates that the warmup epoch choice is not a sensitive hyperparameter.
# Tests tau_warmup ∈ {10, 20, 30, 40} and reports best accuracy.
# Required by Q1 reviewers examining hyperparameter sensitivity.
# =============================================================================

# NOTE: Run only after the main experiment is confirmed successful.
# Each variant takes the same wall-clock time as a full run.
# Recommended: run on CIFAR-100 only for this analysis.

TAU_WARMUP_VALUES = [10, 20, 30, 40]
tau_sensitivity_results = {}

for tw in TAU_WARMUP_VALUES:
    cfg_variant = dict(CONFIG)  # copy
    cfg_variant['tau_warmup']  = tw
    cfg_variant['experiment']  = f'ProCAMMix_tauWarmup{tw}'
    print(f"\nRunning tau_warmup={tw}...")
    hist, best = run_experiment(cfg_variant)
    tau_sensitivity_results[tw] = best

print("\n── Tau Warmup Sensitivity ──")
for tw, acc in tau_sensitivity_results.items():
    print(f"  tau_warmup={tw:2d}  →  best_top1={acc:.2f}%")

In [4]:
# ============================================================
# SELF-CONTAINED ABLATION CELL
# No dependencies. Paste and run directly.
# Runtime: ~2.5 hours on T4
# ============================================================
import os, json, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader

# ── Fixed settings ───────────────────────────────────────────
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED        = 42
EPOCHS      = 50
BATCH       = 128
NUM_CLASSES = 100
RESULTS_DIR = '/content/drive/MyDrive/ProCAMMix/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# ── Data ─────────────────────────────────────────────────────
def get_data():
    mean = (0.5071, 0.4867, 0.4408)
    std  = (0.2675, 0.2565, 0.2761)
    tr = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)])
    te = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)])
    tr_ds = torchvision.datasets.CIFAR100('/content/data', True,  download=True, transform=tr)
    te_ds = torchvision.datasets.CIFAR100('/content/data', False, download=True, transform=te)
    tr_dl = DataLoader(tr_ds, BATCH, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
    te_dl = DataLoader(te_ds, BATCH, shuffle=False, num_workers=2, pin_memory=True)
    return tr_dl, te_dl

# ── Model: ResNet-18 modified for CIFAR ──────────────────────
def get_model():
    m         = models.resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(512, NUM_CLASSES)
    nn.init.kaiming_uniform_(m.fc.weight)
    return m.to(DEVICE)

# ── CAM (ResNet-18 layer4 hook) ───────────────────────────────
def register_cam_hook(model):
    feat = [None]
    def fn(m, i, o): feat[0] = o.detach()
    handle = model.layer4.register_forward_hook(fn)
    return feat, handle

def get_cam(model, feat, labels):
    weights_fc = model.fc.weight[labels]          # (B, 512) - Renamed to avoid collision
    f   = feat[0]                          # (B, 512, h, w)
    B,C,h,f_w = f.shape                    # Renamed 'w' to 'f_w'
    cam = F.relu((weights_fc.view(B,C,1,1)*f).sum(1)) # Use weights_fc here
    mn  = cam.view(B,-1).min(1)[0].view(B,1,1)
    mx  = cam.view(B,-1).max(1)[0].view(B,1,1)
    return (cam - mn) / (mx - mn + 1e-8)

# ── EMA buffer ────────────────────────────────────────────────
class EMABuf:
    def __init__(self, cam_h, cam_w):
        self.buf  = torch.zeros(NUM_CLASSES, cam_h, cam_w)
        self.seen = torch.zeros(NUM_CLASSES, dtype=torch.bool)
    def update(self, cams, labels):
        for i in range(len(labels)):
            c = labels[i].item()
            if not self.seen[c]:
                self.buf[c] = cams[i]; self.seen[c] = True
            else:
                self.buf[c] = 0.9*self.buf[c] + 0.1*cams[i]
    def get(self, c):
        return self.buf[c] if self.seen[c] else torch.ones(self.buf.shape[1], self.buf.shape[2])

# ── Augmentation ──────────────────────────────────────────────
def augment(images, labels, model, feat, ema, tau, epoch,
            no_progressive, no_labelcorr, no_ema):
    eff_tau = 1.0 if no_progressive else tau
    B,C,H,W = images.shape

    model.eval()
    with torch.no_grad():
        _ = model(images)
        cams = get_cam(model, feat, labels)
    model.train()

    if not no_ema:
        ema.update(cams.cpu(), labels.cpu())

    idx      = torch.randperm(B, device=DEVICE)
    labels_b = labels[idx]
    lams     = np.random.beta(1.0, 1.0, size=B)

    cams_j = cams[idx].cpu() if no_ema else \
             torch.stack([ema.get(labels_b[i].item()) for i in range(B)])

    cams_up = F.interpolate(
        cams_j.unsqueeze(1).float(),
        size=(H,W), mode='bilinear', align_corners=False).squeeze(1)

    cut_h = np.maximum(1,(H*np.sqrt(np.maximum(0,1-lams))).astype(int))
    cut_w = np.maximum(1,(W*np.sqrt(np.maximum(0,1-lams))).astype(int))

    mixed = images.clone(); lam_list = []

    for i in range(B):
        cam_i = cams_up[i]
        if eff_tau < 1e-6 or cam_i.sum() < 1e-6:
            cy,cx = random.randint(0,H-1), random.randint(0,W-1)
        else:
            bl   = eff_tau*cam_i + (1-eff_tau)*torch.ones_like(cam_i)
            bl   = bl/bl.sum()
            flat = bl.flatten().cpu().numpy().astype(np.float64)
            flat/= flat.sum()
            ip   = np.random.choice(len(flat), p=flat)
            cy,cx= divmod(ip, W)
        x1=max(0,cx-cut_w[i]//2); x2=min(W,x1+cut_w[i])
        y1=max(0,cy-cut_h[i]//2); y2=min(H,y1+cut_h[i])
        mixed[i,:,y1:y2,x1:x2] = images[idx[i],:,y1:y2,x1:x2]
        la = 1-(x2-x1)*(y2-y1)/(H*W+1e-8)
        if no_labelcorr:
            lf = la
        else:
            ms = cam_i[y1:y2,x1:x2].sum()/(cam_i.sum()+1e-8)
            lf = (1-eff_tau)*la + eff_tau*(1-ms.item())
        lam_list.append(float(np.clip(lf,0,1)))

    lam_t = torch.tensor(lam_list, dtype=torch.float32, device=DEVICE)
    return mixed, labels, labels_b, lam_t

# ── Train / Eval ──────────────────────────────────────────────
def get_tau(ep, total=50, warmup=20):
    if ep < warmup: return 0.0
    return (ep - warmup) / max(1, total - warmup)

def get_tau(ep, total=EPOCHS, warmup=10):
    if ep < warmup: return 0.0
    return (ep - warmup) / max(1, total - warmup)

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); correct = total = 0
    for x,y in dl:
        x,y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x).argmax(1)
        correct += pred.eq(y).sum().item(); total += y.size(0)
    model.train()
    return 100.0*correct/total

def run(name, no_progressive, no_labelcorr, no_ema):
    print(f"\n{'='*50}\n  {name}\n{'='*50}")
    set_seed(SEED)
    train_dl, test_dl = get_data()
    model = get_model()
    feat, handle = register_cam_hook(model)

    # Determine cam_h and cam_w dynamically from model's output or set based on known architecture
    # For ResNet-18 with 32x32 input and CIFAR modifications, layer4 output is 4x4 spatial.
    cam_h_val = 4
    cam_w_val = 4
    ema   = EMABuf(cam_h_val, cam_w_val)

    opt = torch.optim.SGD(model.parameters(), lr=0.1,
                          momentum=0.9, weight_decay=1e-4, nesterov=True)
    def lr_fn(ep):
        if ep < 5: return (ep+1)/5
        p = (ep-5)/max(1,EPOCHS-5)
        return 1e-4/0.1 + (1-1e-4/0.1)*0.5*(1+math.cos(math.pi*p))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

    best = 0.0
    for ep in range(EPOCHS):
        model.train()
        tau = get_tau(ep)
        for imgs, lbls in train_dl:
            imgs,lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            mix,ya,yb,lam = augment(imgs, lbls, model, feat, ema, tau, ep,
                                     no_progressive, no_labelcorr, no_ema)
            opt.zero_grad()
            out  = model(mix)
            la   = F.cross_entropy(out, ya, reduction='none')
            lb   = F.cross_entropy(out, yb, reduction='none')
            loss = (lam*la + (1-lam)*lb).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        sched.step()
        acc = evaluate(model, test_dl)
        if acc > best: best = acc
        if ep % 10 == 0 or ep == EPOCHS-1:
            print(f"  ep{ep:3d} | test={acc:.2f}% | best={best:.2f}% | τ={tau:.2f}")

    handle.remove()
    print(f"\n  ✓ {name}: {best:.2f}%")
    return best

# ── Run all four variants ─────────────────────────────────────
results = {}
results['Full_ProCAMMix']   = run('Full ProCAMMix (reference)', False, False, False)
results['A1_NoProgressive'] = run('A1 — No Progressive Schedule', True,  False, False)
results['A2_NoLabelCorr']   = run('A2 — No Label Correction',     False, True,  False)
results['A3_NoEMA']         = run('A3 — No EMA Buffer',           False, False, True)

# ── Print table ───────────────────────────────────────────────
ref = results['Full_ProCAMMix']
print(f"\n{'─'*52}")
print(f"  ABLATION STUDY  |  CIFAR-100, ResNet-18, 50 epochs")
print(f"{'─'*52}")
for k,v in results.items():
    delta = f"{v-ref:+.2f}%" if k != 'Full_ProCAMMix' else "reference"
    print(f"  {k:<25s}: {v:.2f}%  ({delta})")
print(f"{'─'*52}")

with open(f"{RESULTS_DIR}/ablation_final.json",'w') as f:
    json.dump(results, f, indent=2)
print(f"\nSaved to {RESULTS_DIR}/ablation_final.json")


  Full ProCAMMix (reference)
  ep  0 | test=17.34% | best=17.34% | τ=0.00
  ep 10 | test=54.02% | best=54.02% | τ=0.00
  ep 20 | test=62.96% | best=63.53% | τ=0.25
  ep 30 | test=70.50% | best=70.50% | τ=0.50
  ep 40 | test=75.16% | best=75.16% | τ=0.75
  ep 49 | test=76.67% | best=76.67% | τ=0.97

  ✓ Full ProCAMMix (reference): 76.67%

  A1 — No Progressive Schedule
  ep  0 | test=18.39% | best=18.39% | τ=0.00
  ep 10 | test=54.93% | best=54.93% | τ=0.00
  ep 20 | test=63.80% | best=63.80% | τ=0.25
  ep 30 | test=70.80% | best=70.80% | τ=0.50
  ep 40 | test=74.93% | best=74.93% | τ=0.75
  ep 49 | test=76.68% | best=76.75% | τ=0.97

  ✓ A1 — No Progressive Schedule: 76.75%

  A2 — No Label Correction
  ep  0 | test=17.34% | best=17.34% | τ=0.00
  ep 10 | test=54.02% | best=54.02% | τ=0.00
  ep 20 | test=62.55% | best=62.72% | τ=0.25
  ep 30 | test=70.55% | best=70.55% | τ=0.50
  ep 40 | test=75.10% | best=75.10% | τ=0.75
  ep 49 | test=76.54% | best=76.61% | τ=0.97

  ✓ A2 — No Label